**Integrantes**

Andrey Crence Fernandes - 573840

Arthur de Oliveira Carvalho - 573499

Gabriel Henrique Silva de Melo Rodrigues - 573093

In [17]:
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to r

In [18]:
import subprocess
import time

subprocess.Popen(["ollama", "serve"])

time.sleep(5)

In [19]:
!ollama serve > /dev/null 2>&1 &
!until curl -s http://localhost:11434 > /dev/null; do sleep 1; done
!ollama pull llama3.2:1b

In [20]:
!pip install ollama -q

In [21]:
import random
import ollama

def gerar_dados():
    temperatura = random.randint(10, 120)

    energia = random.randint(0, 100)

    comunicacao = random.choice([
        "Estável",
        "Instável",
        "Perdida"
    ])

    return temperatura, energia, comunicacao

def gerar_modulos():

    estados = [
        "OK",
        "ALERTA",
        "CRÍTICO",
    ]

    return {
        "Motores": random.choice(estados),
        "Habitação": random.choice(estados),
        "Pesquisa": random.choice(estados),
        "Painéis Solares": random.choice(estados)
    }

def analisar_alertas(
    temperatura,
    energia,
    comunicacao,
    modulos
):

    alertas = []

    if temperatura > 100:
        alertas.append("Temperatura Crítica")

    elif temperatura > 80:
        alertas.append("Temperatura Elevada")

    if energia < 20:
        alertas.append("Energia Crítica")

    if comunicacao == "Instável":
        alertas.append("Comunicação Instável")

    elif comunicacao == "Perdida":
        alertas.append("Comunicação Perdida")

    for nome, estado in modulos.items():

        if estado == "CRÍTICO":
          alertas.append(
            f"Módulo {nome} em Estado Crítico"
        )

        elif estado == "ALERTA":
          alertas.append(
            f"Módulo {nome} em Alerta"
        )

    return alertas


def tomar_decisao(
    temperatura,
    energia,
    comunicacao,
    modulos
):

    acoes = []

    if temperatura > 100:
        acoes.append(
            "Sistema de Resfriamento Ativado"
        )

    elif temperatura > 80:
        acoes.append(
            "Monitoramento Reforçado da Temperatura"
    )

    if energia < 20:
        acoes.append(
            "Modo Economia de Energia Ativado"
        )

    if comunicacao == "Perdida":
        acoes.append(
            "Antena Reserva Ativada"
        )

    elif comunicacao == "Instável":
        acoes.append(
            "Verificação dos Sistemas de Comunicação"
    )

    for nome, estado in modulos.items():

        if estado == "CRÍTICO":
          acoes.append(
              f"Equipe Técnica Acionada para {nome}"
            )

        elif estado == "ALERTA":
          acoes.append(
              f"Monitoramento Reforçado do módulo {nome}"
        )

    if not acoes:
        acoes.append(
            "Operação Normal"
        )

    return acoes


def definir_status(alertas):

    for alerta in alertas:

        texto = alerta.lower()

        if (
            "crítica" in texto
            or "crítico" in texto
            or "perdida" in texto
        ):
            return "🔴 CRÍTICO"

    if len(alertas) > 0:
        return "🟡 ATENÇÃO"

    return "🟢 NORMAL"


def barra_energia(valor):

    tamanho = 20

    preenchido = int(
        (valor / 100) * tamanho
    )

    return (
        "█" * preenchido
        + "░" * (tamanho - preenchido)
    )


def analisar_com_ia(
    status,
    alertas,
    acoes
):

    prompt = f"""
Status da missão:
{status}

Alertas:
{', '.join(alertas)}

Ações automáticas:
{', '.join(acoes)}

Gere um resumo operacional curto.

Explique as seguintes coisas:
1. Liste quais alertas justificam o status atual.
2. Alertas que exigem atenção.
3. Como as ações executadas ajudam a missão.

Use somente as informações fornecidas.
Não invente causas, defeitos, acidentes ou consequências.
Responda em até 12 frases.
"""

    resposta = ollama.chat(
        model="llama3.2:1b",
        messages=[
            {
                "role": "system",
    "content": """
Você é um assistente de missão espacial.

Receberá um relatório já analisado.

NÃO faça cálculos.
NÃO interprete valores numéricos.
NÃO reclassifique a missão.
NÃO invente novos riscos.

Apenas explique os alertas informados pelo sistema
e apresente recomendações curtas.

Responda em no máximo 5 linhas.
"""
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return resposta["message"]["content"]

temperatura, energia, comunicacao = gerar_dados()

modulos = gerar_modulos()

alertas = analisar_alertas(
    temperatura,
    energia,
    comunicacao,
    modulos
)

acoes = tomar_decisao(
    temperatura,
    energia,
    comunicacao,
    modulos
)

status = definir_status(alertas)

analise_ia = analisar_com_ia(
    status,
    alertas,
    acoes
)

print("=" * 60)
print("                  SKYGUARD AI")
print("=" * 60)

print("\nDADOS DA MISSÃO\n")

print(
    f"Temperatura : {temperatura}°C"
)

print(
    f"Energia     : "
    f"{barra_energia(energia)} "
    f"{energia}%"
)

print(
    f"Comunicação : {comunicacao}"
)

print("\nMÓDULOS DA MISSÃO\n")

for nome, estado in modulos.items():
    print(f"{nome:<16}: {estado}")

print("\n" + "-" * 60)

print("\nALERTAS\n")

if alertas:

    for alerta in alertas:
        print(f"[!] {alerta}")

else:
    print("[✓] Nenhum alerta detectado")

print("\n" + "-" * 60)

print("\nAÇÕES AUTOMÁTICAS\n")

for acao in acoes:
    print(f"[✓] {acao}")

print("\n" + "-" * 60)

print("\nSTATUS GERAL\n")

print(status)

print("\n" + "-" * 60)

print("\nANÁLISE DA IA\n")

print(analise_ia)

print("\n" + "=" * 60)
print("              FIM DO RELATÓRIO")
print("=" * 60)

                  SKYGUARD AI

DADOS DA MISSÃO

Temperatura : 112°C
Energia     : ░░░░░░░░░░░░░░░░░░░░ 3%
Comunicação : Instável

MÓDULOS DA MISSÃO

Motores         : ALERTA
Habitação       : CRÍTICO
Pesquisa        : CRÍTICO
Painéis Solares : OK

------------------------------------------------------------

ALERTAS

[!] Temperatura Crítica
[!] Energia Crítica
[!] Comunicação Instável
[!] Módulo Motores em Alerta
[!] Módulo Habitação em Estado Crítico
[!] Módulo Pesquisa em Estado Crítico

------------------------------------------------------------

AÇÕES AUTOMÁTICAS

[✓] Sistema de Resfriamento Ativado
[✓] Modo Economia de Energia Ativado
[✓] Verificação dos Sistemas de Comunicação
[✓] Monitoramento Reforçado do módulo Motores
[✓] Equipe Técnica Acionada para Habitação
[✓] Equipe Técnica Acionada para Pesquisa

------------------------------------------------------------

STATUS GERAL

🔴 CRÍTICO

------------------------------------------------------------

ANÁLISE DA IA

**Status at